# Advanced RAG — SEC Filings

**Pipeline**: Metadata Extraction → Query Rewriting → Multi-Query → Hybrid Search (BM25 + Dense + RRF) → CrossEncoder Reranking → Context Compression → LLM Generation with Citations

## What's new vs Baseline RAG?

| Stage | Baseline | Advanced |
|-------|----------|----------|
| Query prep | Raw query | Rewrite + 2 variants |
| Retrieval | Dense only | BM25 + Dense + RRF |
| Scope | All chunks | Metadata-filtered |
| Scoring | Cosine sim | CrossEncoder rerank |
| Context | Full chunks | Sentence-level compression |
| Answer | Plain text | Text with [n] citations |

Each improvement is independently motivated:
1. **Metadata filtering** — restricts search to the relevant company/year, reducing irrelevant noise
2. **Query rewriting** — aligns query vocabulary with financial document language
3. **Multi-query** — different phrasings capture different relevant chunks
4. **Hybrid BM25 + Dense** — BM25 handles exact financial term matching; dense handles semantic similarity
5. **CrossEncoder reranking** — more accurate relevance scoring than bi-encoder cosine similarity
6. **Context compression** — removes off-topic sentences, reduces LLM context noise
7. **Citation generation** — improves traceability and faithfulness

In [1]:
print('hi')

hi


In [2]:
# Uncomment to install dependencies
# !pip install sentence-transformers chromadb rank-bm25 groq python-dotenv pandas tqdm

## 1. Configuration

In [43]:
CONFIG = {
    # --- Paths (relative to this notebook) ---
    "chroma_db_path": "../sec_rag_team_share/chroma_db",
    "chunks_path":    "../sec_rag_team_share/sec_data/chunks/sec_chunks.jsonl",
    "eval_path":      "../sec_rag_team_share/evaluation/GenAI Eval QA.csv",
    "results_dir":    "./results",

    # --- Models ---
    "dense_model_name":   "sentence-transformers/all-mpnet-base-v2",
    "reranker_model_name":"cross-encoder/ms-marco-MiniLM-L-6-v2",
    # Execution profile: 'dev' or 'final' (same model — kept for parity with Baseline RAG)
    "execution_profile":  "dev",
    "gemini_dev_model":   "gemini-2.5-flash-lite",
    "gemini_final_model": "gemini-2.5-flash-lite",
    "agent_model":        "gemini-2.5-flash-lite",  # query rewriting / multi-query
    "judge_model":        "gemini-2.5-flash-lite",

    # --- Retrieval (matched to CrewAI for fair comparison) ---
    "bm25_top_k":        15,
    "dense_top_k":       15,
    "rerank_top_k":      7,
    "multi_query_n":     2,   # number of additional query variants to generate
    "rrf_k":             60,  # RRF smoothing constant

    # --- Context compression ---
    "compress_top_sentences": 4,  # sentences to keep per chunk; None to disable

    # --- Generation ---
    "max_context_chars":     9000,
    "temperature_rewriter":  0.2,
    "temperature_generator": 0.2,
    "temperature_judge":     0.1,   # 0.0 causes empty responses in Gemini via LiteLLM
    "max_tokens_answer":     512,
    "max_tokens_judge":      256,

    # --- Cost tracking (gemini-2.5-flash-lite) ---
    "gemini_cost_input_per_1m":  0.10,
    "gemini_cost_output_per_1m": 0.40,

    # --- Evaluation ---
    "eval_split": "dev",

    # --- Rate limiting (Gemini free tier: 10 RPM) ---
    # Advanced RAG makes ~6 LLM calls per question: rewrite + 2 variants + generate + judge_corr + judge_faith
    "max_rpm":                    10,
    "inter_question_sleep_sec":   40,
    "llm_max_retries":            3,
    "llm_retry_base_delay_sec":   5,
}

## 2. Imports & Setup

In [44]:
import os
import re
import json
import time
import threading
from collections import deque, Counter as _Counter
from pathlib import Path
from typing import Optional
import re as _re

import numpy as np
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from pydantic import BaseModel, Field

import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from google import genai
from google.genai import types as genai_types

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY in your .env file"

LLM_MODEL = (
    CONFIG["gemini_final_model"]
    if CONFIG["execution_profile"] == "final"
    else CONFIG["gemini_dev_model"]
)
print(f"Execution profile : {CONFIG['execution_profile']}")
print(f"LLM model         : {LLM_MODEL}")

Execution profile : dev
LLM model         : gemini-2.5-flash-lite


## 3. Load Models

In [45]:
print("Loading embedding model...")
embed_model = SentenceTransformer(CONFIG["dense_model_name"])
print(f"  ok {CONFIG['dense_model_name']}")

print("Loading reranker...")
reranker = CrossEncoder(CONFIG["reranker_model_name"])
print(f"  ok {CONFIG['reranker_model_name']}")

genai_client = genai.Client(api_key=GEMINI_API_KEY)
print(f"  ok Gemini client ready (model: {LLM_MODEL})")


# ── Rate Limiter ───────────────────────────────────────────────────────────────
class RateLimiter:
    def __init__(self, max_rpm: int):
        self.max_rpm = max_rpm
        self._ts: deque = deque()
        self._lock = threading.Lock()

    def wait(self):
        with self._lock:
            now = time.time()
            while self._ts and now - self._ts[0] >= 60.0:
                self._ts.popleft()
            if len(self._ts) >= self.max_rpm:
                sleep_for = 60.0 - (now - self._ts[0])
                if sleep_for > 0:
                    print(f"  [RateLimit] sleeping {sleep_for:.1f}s ...")
                    time.sleep(sleep_for)
            self._ts.append(time.time())

rate_limiter = RateLimiter(CONFIG["max_rpm"])


# ── Per-Question Token Accumulator ─────────────────────────────────────────────
_question_tokens: dict = {}

def _reset_question_tokens() -> None:
    global _question_tokens
    _question_tokens = {}

def _add_tokens(step: str, input_tok: int, output_tok: int) -> None:
    if step not in _question_tokens:
        _question_tokens[step] = {'input': 0, 'output': 0}
    _question_tokens[step]['input']  += int(input_tok  or 0)
    _question_tokens[step]['output'] += int(output_tok or 0)

def _get_question_token_totals() -> tuple:
    total_in  = sum(v['input']  for v in _question_tokens.values())
    total_out = sum(v['output'] for v in _question_tokens.values())
    return total_in, total_out

def _estimate_cost_usd(total_input: int, total_output: int) -> float:
    rate_in  = CONFIG.get('gemini_cost_input_per_1m',  0.10) / 1_000_000
    rate_out = CONFIG.get('gemini_cost_output_per_1m', 0.40) / 1_000_000
    return total_input * rate_in + total_output * rate_out


# ── LLM Call ───────────────────────────────────────────────────────────────────
def call_llm(
    messages: list,
    model: str = None,
    temperature: float = 0.2,
    max_tokens: int = 512,
    json_mode: bool = False,
    step: str = 'generate',
) -> str:
    """Call Gemini with retry. Accumulates tokens to _question_tokens[step]."""
    model = model or LLM_MODEL
    system_instruction = None
    user_parts = []
    for msg in messages:
        if msg["role"] == "system":
            system_instruction = msg["content"]
        else:
            user_parts.append(msg["content"])
    contents = "\n\n".join(user_parts) if user_parts else ""

    cfg_kwargs = dict(temperature=temperature, max_output_tokens=max_tokens)
    if json_mode:
        cfg_kwargs["response_mime_type"] = "application/json"
    if system_instruction:
        cfg_kwargs["system_instruction"] = system_instruction

    for attempt in range(CONFIG["llm_max_retries"]):
        try:
            rate_limiter.wait()
            resp = genai_client.models.generate_content(
                model=model,
                contents=contents,
                config=genai_types.GenerateContentConfig(**cfg_kwargs),
            )
            if step and resp.usage_metadata:
                _add_tokens(step,
                    resp.usage_metadata.prompt_token_count     or 0,
                    resp.usage_metadata.candidates_token_count or 0)
            return resp.text.strip()
        except Exception as e:
            delay = CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt)
            print(f"  [WARN] attempt {attempt+1}/{CONFIG['llm_max_retries']} failed ({e}). Retrying in {delay}s...")
            time.sleep(delay)
    raise RuntimeError(f"LLM call failed after {CONFIG['llm_max_retries']} attempts")


# ── Heuristic Scoring Utilities ────────────────────────────────────────────────
def _tokenize(text: str) -> list:
    return _re.sub(r'[^\w\s]', '', text.lower()).split()

def token_f1_score(pred: str, ref: str) -> float:
    if not pred or not ref:
        return 0.0
    pred_toks = _tokenize(pred)
    ref_toks  = _tokenize(ref)
    if not pred_toks or not ref_toks:
        return 0.0
    common = sum((_Counter(pred_toks) & _Counter(ref_toks)).values())
    if common == 0:
        return 0.0
    precision = common / len(pred_toks)
    recall    = common / len(ref_toks)
    return 2 * precision * recall / (precision + recall)

def numerical_accuracy_score(pred: str, ref: str) -> Optional[float]:
    nums_ref  = [float(n.replace(',', '')) for n in _re.findall(r'-?\d[\d,]*\.?\d*', ref)]
    nums_pred = [float(n.replace(',', '')) for n in _re.findall(r'-?\d[\d,]*\.?\d*', pred)]
    if not nums_ref:
        return None
    hits = sum(
        1 for r in nums_ref
        if any(abs(p - r) / (abs(r) + 1e-9) < 0.01 for p in nums_pred)
    )
    return hits / len(nums_ref)

def compute_decision_from_text(answer_text: str) -> str:
    lower = answer_text.lower()
    refusal_phrases = [
        'insufficient', 'not contain', 'not available', 'cannot find',
        "don't have", 'no information', 'not provided', 'unable to',
        'not enough', 'not present', 'not found', 'insufficient data',
    ]
    return 'refuse' if any(p in lower for p in refusal_phrases) else 'answer'


# ── Structured Judge (matches CrewAI) ─────────────────────────────────────────
class JudgeOutput(BaseModel):
    score:          float = Field(default=0.0, description='Score 0.0-1.0')
    claims_covered: float = Field(default=0.0, description='Fraction of key facts covered 0.0-1.0')
    reason:         str   = Field(default='',  description='Short explanation')

def _build_correctness_prompt(question: str, reference_answer: str, candidate_answer: str) -> str:
    return (
        'Score the candidate answer against the reference answer from 0 to 1 for a financial QA task. '
        '1 = correct value, correct units, correct period. '
        '0 = wrong number, wrong company, wrong period, or completely missing. '
        'Give partial credit for answers close but with unit errors. '
        'Also set claims_covered: fraction of distinct facts/numbers in the reference present in the candidate. '
        'Return a valid JSON object only that matches the requested schema.\n\n'
        f'Question:\n{question}\n\n'
        f'Reference answer:\n{reference_answer}\n\n'
        f'Candidate answer:\n{candidate_answer}'
    )

def _build_faithfulness_prompt(question: str, context: str, candidate_answer: str) -> str:
    return (
        'Score how well the candidate answer is grounded in the retrieved filing context from 0 to 1. '
        '1 = every number and claim is directly supported by the context. '
        '0 = answer contains numbers or claims not present in the context (hallucinated). '
        'Also set claims_covered: fraction of claims in the candidate supported by the context. '
        'Return a valid JSON object only that matches the requested schema.\n\n'
        f'Question:\n{question}\n\n'
        f'Retrieved context:\n{context}\n\n'
        f'Candidate answer:\n{candidate_answer}'
    )

def _safe_judge_call(prompt_text: str, task_name: str) -> JudgeOutput:
    _fb = JudgeOutput(score=0.0, claims_covered=0.0, reason='judge fallback — all attempts failed')
    for attempt in range(CONFIG["llm_max_retries"]):
        try:
            rate_limiter.wait()
            response = genai_client.models.generate_content(
                model=CONFIG["judge_model"],
                contents=prompt_text,
                config=genai_types.GenerateContentConfig(
                    response_mime_type='application/json',
                    response_schema=JudgeOutput,
                    temperature=CONFIG.get('temperature_judge', 0.1),
                ),
            )
            if response.usage_metadata:
                _add_tokens(task_name,
                    response.usage_metadata.prompt_token_count     or 0,
                    response.usage_metadata.candidates_token_count or 0)
            result = response.parsed
            if result is None:
                result = JudgeOutput(**json.loads(response.text))
            return result
        except Exception as e:
            print(f"  [WARN] Judge ({task_name}) attempt {attempt+1} failed: {e}")
            if attempt < CONFIG["llm_max_retries"] - 1:
                time.sleep(CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt))
    return _fb

def llm_judge_correctness(question: str, reference_answer: str, candidate_answer: str) -> tuple:
    result = _safe_judge_call(
        _build_correctness_prompt(question, reference_answer, candidate_answer),
        task_name='judge_correctness',
    )
    return (max(0.0, min(1.0, float(result.score))),
            max(0.0, min(1.0, float(result.claims_covered))),
            result.reason)

def llm_judge_faithfulness(question: str, context: str, candidate_answer: str) -> tuple:
    result = _safe_judge_call(
        _build_faithfulness_prompt(question, context[:CONFIG.get('max_context_chars', 9000)], candidate_answer),
        task_name='judge_faithfulness',
    )
    return (max(0.0, min(1.0, float(result.score))),
            max(0.0, min(1.0, float(result.claims_covered))),
            result.reason)

print("Client, rate limiter, token tracking, heuristics, and judge ready.")

Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5724.50it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ok sentence-transformers/all-mpnet-base-v2
Loading reranker...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4458.01it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ok cross-encoder/ms-marco-MiniLM-L-6-v2
  ok Gemini client ready (model: gemini-2.5-flash-lite)
Client, rate limiter, token tracking, heuristics, and judge ready.


## 4. Load Data

In [46]:
# --- ChromaDB ---
print("Connecting to ChromaDB...")
chroma_client = chromadb.PersistentClient(path=CONFIG["chroma_db_path"])
collections = chroma_client.list_collections()
collection = chroma_client.get_collection(collections[0].name)
print(f"  ok '{collections[0].name}'  ({collection.count():,} chunks)")

# --- JSONL chunks (needed for BM25) ---
print("Loading SEC chunks from JSONL...")
raw_chunks = []
with open(CONFIG["chunks_path"], "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_chunks.append(json.loads(line))

chunks_df = pd.DataFrame(raw_chunks)

# Filter low-value chunks (matching the existing pipeline)
if "flag_low_value_combined" in chunks_df.columns:
    before = len(chunks_df)
    chunks_df = chunks_df[~chunks_df["flag_low_value_combined"].fillna(False)].reset_index(drop=True)
    print(f"  Filtered {before - len(chunks_df)} low-value chunks")

print(f"  ok {len(chunks_df):,} chunks loaded")


def make_contextual_chunk(row) -> str:
    """Reconstruct the contextual chunk text (must match what was stored in ChromaDB)."""
    company = row.get("company_name", row.get("company", "?"))
    ticker  = row.get("ticker", "?")
    form    = row.get("form_type", "?")
    date    = str(row.get("filing_date", "?"))[:10]
    year    = row.get("filing_year", "?")
    section = row.get("section_title", "?")
    text    = row.get("text", row.get("raw_chunk", ""))
    return (
        f"Company: {company} ({ticker})\n"
        f"Filing: {form} | Date: {date} | Year: {year}\n"
        f"Section: {section}\n"
        f"Content: {text}"
    )


# Build contextual chunks column and BM25 tokens
chunks_df["contextual_chunk"] = chunks_df.apply(make_contextual_chunk, axis=1)
chunks_df["bm25_tokens"] = chunks_df["contextual_chunk"].str.lower().str.split()

# Build BM25 index
print("Building BM25 index...")
bm25_index = BM25Okapi(chunks_df["bm25_tokens"].tolist())
print(f"  ok BM25 index built  ({len(chunks_df):,} documents)")

# --- Adjacent-chunk expansion lookup ---
# Maps chunk_id string -> DataFrame row index (for expand_adjacent)
_chunk_id_to_row = {str(row["chunk_id"]): idx for idx, row in chunks_df.iterrows()}

# Maps (ticker, form_type, filing_date) -> {chunk_index -> df_row_idx}
_filing_chunk_lookup: dict = {}
for idx, row in chunks_df.iterrows():
    key = (str(row["ticker"]), str(row["form_type"]), str(row.get("filing_date", ""))[:10])
    ci  = int(row.get("chunk_index", -1))
    if ci >= 0:
        _filing_chunk_lookup.setdefault(key, {})[ci] = idx

print(f"  ok adjacent-chunk lookup built  ({len(_filing_chunk_lookup)} filings)")

Connecting to ChromaDB...
  ok 'sec_filings'  (12,725 chunks)
Loading SEC chunks from JSONL...
  Filtered 550 low-value chunks
  ok 12,725 chunks loaded
Building BM25 index...
  ok BM25 index built  (12,725 documents)
  ok adjacent-chunk lookup built  (144 filings)


## 5. Step 1 — Metadata Extraction

Extract company ticker, filing year, and form type from the query to narrow retrieval scope.
This reduces the search space from ~12,600 chunks to only the relevant company/period.

In [47]:
# Known tickers in the dataset
KNOWN_TICKERS = {
    "AAPL", "MSFT", "NVDA", "JPM", "BAC", "BRK.B", "BRK",
    "JNJ", "UNH", "XOM", "CVX", "WMT", "COST",
}

# Company name -> ticker mapping for natural language mentions
COMPANY_TO_TICKER = {
    "apple": "AAPL", "microsoft": "MSFT", "nvidia": "NVDA",
    "jpmorgan": "JPM", "jp morgan": "JPM", "bank of america": "BAC",
    "berkshire": "BRK.B", "johnson": "JNJ", "unitedhealth": "UNH",
    "exxon": "XOM", "exxonmobil": "XOM", "chevron": "CVX",
    "walmart": "WMT", "costco": "COST",
}


def extract_metadata(query: str) -> dict:
    """
    Extract ticker, filing_year, and form_type from query text.
    Uses regex for year/form_type and a lookup table for company names.
    Returns dict with None values for fields not found.
    """
    q_upper = query.upper()
    q_lower = query.lower()

    # --- Ticker ---
    ticker = None
    for t in KNOWN_TICKERS:
        if re.search(r"\b" + re.escape(t) + r"\b", q_upper):
            ticker = t
            break
    if ticker is None:
        for name, t in COMPANY_TO_TICKER.items():
            if name in q_lower:
                ticker = t
                break

    # --- Filing year (2020-2029) ---
    year_match = re.search(r"\b(202[0-9])\b", query)
    filing_year = int(year_match.group(1)) if year_match else None

    # --- Form type ---
    form_type = None
    if re.search(r"\b10-K\b|\bannual report\b|\bannual filing\b", query, re.IGNORECASE):
        form_type = "10-K"
    elif re.search(r"\b10-Q\b|\bquarterly report\b|\bquarterly filing\b", query, re.IGNORECASE):
        form_type = "10-Q"

    return {"ticker": ticker, "filing_year": filing_year, "form_type": form_type}


# Quick test
print(extract_metadata("What was Apple's revenue in fiscal 2024 10-K?"))
print(extract_metadata("Compare NVDA and MSFT operating income for 2023"))

{'ticker': 'AAPL', 'filing_year': 2024, 'form_type': '10-K'}
{'ticker': 'MSFT', 'filing_year': 2023, 'form_type': None}


## 6. Step 2 — Query Rewriting

Rewrite the user's question into a retrieval-optimised query.
Financial questions often use colloquial phrasing ("how much did they make") that doesn't match
document language ("net revenue", "operating income"). A single LLM call fixes the vocabulary mismatch.

In [48]:
REWRITE_SYSTEM = (
    "You are a search query optimizer for SEC filings (10-K, 10-Q). "
    "Rewrite the user's question as a concise retrieval query that maximizes recall of relevant "
    "financial document chunks. Use precise financial terminology (e.g., 'net revenue', "
    "'operating income', 'diluted EPS'). Keep it under 25 words. "
    "Return only the rewritten query, nothing else."
)


def rewrite_query(query: str) -> str:
    """Rewrite query for better retrieval alignment with financial document language."""
    return call_llm(
        messages=[
            {"role": "system", "content": REWRITE_SYSTEM},
            {"role": "user",   "content": query},
        ],
        model=CONFIG["agent_model"],
        temperature=CONFIG["temperature_rewriter"],
        max_tokens=80,
    )

## 7. Step 3 — Multi-Query Generation

Generate additional query variants to increase retrieval coverage.
Different phrasings surface different relevant chunks — e.g., asking about "revenue growth"
and "year-over-year revenue change" may retrieve complementary passages.

In [49]:
MULTI_QUERY_SYSTEM = (
    "Generate {n} alternative retrieval queries for the following question about SEC filings. "
    "Each variant should use different financial terminology or focus on a different aspect "
    "of the question to maximise retrieval coverage. "
    'Return a JSON object with key "queries" containing an array of {n} query strings.'
)


def generate_multi_queries(query: str, n: int = None) -> list:
    """Generate n alternative query variants for multi-query retrieval."""
    n = n or CONFIG["multi_query_n"]
    raw = call_llm(
        messages=[
            {"role": "system", "content": MULTI_QUERY_SYSTEM.format(n=n)},
            {"role": "user",   "content": query},
        ],
        model=CONFIG["agent_model"],
        temperature=CONFIG["temperature_rewriter"],
        max_tokens=200,
        json_mode=True,
    )
    try:
        data = json.loads(raw)
        return data.get("queries", [])[:n]
    except Exception:
        return []

## 8. Step 4 — Hybrid Search (BM25 + Dense + RRF)

BM25 excels at exact financial term matching ("EBITDA", "diluted EPS", specific dollar figures).
Dense retrieval excels at semantic similarity.
Reciprocal Rank Fusion (RRF) combines both ranked lists without requiring score normalisation.
Metadata filtering further restricts results to the relevant company/year.

In [50]:
def build_bm25_mask(ticker=None, filing_year=None, form_type=None) -> np.ndarray:
    """Boolean mask to restrict BM25 search to matching metadata."""
    mask = np.ones(len(chunks_df), dtype=bool)
    if ticker:
        mask &= (chunks_df["ticker"].str.upper() == ticker.upper()).values
    if filing_year:
        mask &= (chunks_df["filing_year"].astype(int) == int(filing_year)).values
    if form_type:
        mask &= (chunks_df["form_type"].str.upper() == form_type.upper()).values
    return mask


def bm25_search(query: str, top_k: int, mask: np.ndarray = None) -> list:
    """BM25 retrieval over contextual chunks with optional metadata mask."""
    tokens = query.lower().split()
    scores = bm25_index.get_scores(tokens)
    if mask is not None:
        scores = scores * mask.astype(float)
    top_idx = np.argsort(scores)[::-1]
    # Keep only positive-scoring results
    top_idx = [i for i in top_idx if scores[i] > 0][:top_k]
    results = []
    for i in top_idx:
        row = chunks_df.iloc[i]
        results.append({
            "text":     row["contextual_chunk"],
            "metadata": {
                "ticker":      row.get("ticker", "?"),
                "form_type":   row.get("form_type", "?"),
                "filing_year": int(row.get("filing_year", 0)),
                "company":     row.get("company_name", row.get("company", "?")),
            },
            "score":    float(scores[i]),
            "chunk_id": str(row.get("chunk_id", i)),
        })
    return results


def dense_search(query: str, top_k: int, ticker=None, filing_year=None, form_type=None) -> list:
    """Dense (semantic) retrieval with optional ChromaDB metadata filtering."""
    query_vec = embed_model.encode(query, normalize_embeddings=True).tolist()

    filters = []
    if ticker:
        filters.append({"ticker": {"$eq": ticker}})
    if filing_year:
        filters.append({"filing_year": {"$eq": int(filing_year)}})
    if form_type:
        filters.append({"form_type": {"$eq": form_type}})

    kwargs = dict(
        query_embeddings=[query_vec],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"],
    )

    if len(filters) == 1:
        kwargs["where"] = filters[0]
    elif len(filters) > 1:
        kwargs["where"] = {"$and": filters}

    results = collection.query(**kwargs)

    chunks = []
    for doc, meta, dist, cid in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
        results["ids"][0],
    ):
        chunks.append({
            "text": doc,
            "metadata": meta,
            "score": float(1.0 - dist),
            "chunk_id": cid,
        })

    return chunks


def rrf_merge(result_lists: list, k: int = None) -> list:
    """
    Reciprocal Rank Fusion over multiple ranked lists.
    Formula: score(d) = sum_i 1 / (k + rank_i(d) + 1)
    """
    k = k or CONFIG["rrf_k"]
    scores = {}
    pool   = {}
    for ranked_list in result_lists:
        for rank, chunk in enumerate(ranked_list):
            cid = chunk["chunk_id"]
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank + 1)
            if cid not in pool:
                pool[cid] = chunk
    merged = []
    for cid in sorted(scores, key=scores.__getitem__, reverse=True):
        c = dict(pool[cid])
        c["score"] = scores[cid]
        merged.append(c)
    return merged


def expand_adjacent(candidates: list, expand_n: int = 1) -> list:
    """
    For each chunk in the RRF pool, add the immediately preceding and following
    chunks within the same filing to the candidate pool (score=0.0).
    These extras enter the CrossEncoder reranking pool — the reranker then decides
    whether they're actually relevant, recovering table headers split from data rows.
    """
    existing_ids = {c["chunk_id"] for c in candidates}
    extras = {}

    for chunk in candidates:
        row_idx = _chunk_id_to_row.get(chunk["chunk_id"])
        if row_idx is None:
            continue
        row     = chunks_df.iloc[row_idx]
        base_ci = int(row.get("chunk_index", -1))
        if base_ci < 0:
            continue
        key    = (str(row["ticker"]), str(row["form_type"]), str(row.get("filing_date", ""))[:10])
        ci_map = _filing_chunk_lookup.get(key, {})

        for delta in range(-expand_n, expand_n + 1):
            if delta == 0:
                continue
            adj_idx = ci_map.get(base_ci + delta)
            if adj_idx is None:
                continue
            adj_row = chunks_df.iloc[adj_idx]
            adj_id  = str(adj_row["chunk_id"])
            if adj_id in existing_ids or adj_id in extras:
                continue
            extras[adj_id] = {
                "text":     adj_row["contextual_chunk"],
                "metadata": {
                    "ticker":      adj_row.get("ticker", "?"),
                    "form_type":   adj_row.get("form_type", "?"),
                    "filing_year": int(adj_row.get("filing_year", 0)),
                    "company":     adj_row.get("company_name", adj_row.get("company", "?")),
                },
                "score":    0.0,
                "chunk_id": adj_id,
            }

    return candidates + list(extras.values())

## 9. Step 5 — CrossEncoder Reranking

The bi-encoder embedding model scores query and document independently (fast but approximate).
The CrossEncoder attends to both simultaneously, giving more accurate relevance scores.
Applied to the top RRF candidates to select the final context.

In [26]:
def rerank(candidates: list, query: str, top_k: int = None) -> list:
    """Rerank candidate chunks using CrossEncoder (query, chunk) scoring."""
    top_k = top_k or CONFIG["rerank_top_k"]
    if not candidates:
        return []
    pairs  = [(query, c["text"]) for c in candidates]
    scores = reranker.predict(pairs, show_progress_bar=False)
    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)
    return sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:top_k]

## 10. Step 6 — Context Compression

Each retrieved chunk may contain irrelevant sentences.
We extract only the most relevant sentences (scored by CrossEncoder against the query),
reducing noise in the LLM context window.

In [51]:
def compress_chunk(chunk: dict, query: str, top_n: int = None) -> dict:
    """
    Extract the most relevant sentences from a chunk using CrossEncoder scoring.
    Keeps the structured header (Company/Filing/Section) and replaces the content
    with only the top-N most relevant sentences.
    """
    top_n = top_n or CONFIG["compress_top_sentences"]
    if top_n is None:
        return chunk  # compression disabled

    text = chunk["text"]

    # Separate header (Company/Filing/Section lines) from content
    if "\nContent:" in text:
        header, _, content = text.partition("\nContent:")
        header = header + "\nContent:"
    else:
        header = ""
        content = text

    # Simple sentence splitting (handles ., !, ?, ;)
    sentences = re.split(r"(?<=[.!?;])\s+", content.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 25]

    if len(sentences) <= top_n:
        return chunk  # already short enough

    # Score each sentence against the query
    pairs  = [(query, s) for s in sentences]
    scores = reranker.predict(pairs, show_progress_bar=False)

    # Keep top-N sentences in original document order
    top_idx = sorted(
        sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_n]
    )
    compressed_content = " ".join(sentences[i] for i in top_idx)

    new_chunk = dict(chunk)
    new_chunk["text"] = (header + " " + compressed_content).strip()
    new_chunk["compressed"] = True
    return new_chunk


def compress_context(chunks: list, query: str) -> list:
    """Compress all chunks in the context."""
    return [compress_chunk(c, query) for c in chunks]

## 11. Step 7 — Generation with Citations

Instructs the LLM to cite source chunks using [n] notation.
This improves traceability and allows users to verify claims against the original filings.

In [52]:
GENERATOR_SYSTEM_ADV = (
    "You are a financial analyst answering questions using only the retrieved filing context.\n"
    "Rules:\n"
    "- Be precise with numbers -- include units (millions, billions, %), fiscal periods, and line item names.\n"
    "- Cite your sources using [n] notation after each claim (e.g., 'Revenue was $394B in FY2024 [1]').\n"
    "- Only use information present in the provided context.\n"
    "- If the context does not contain enough information, say 'Insufficient information in the provided filings.'"
)


def format_context_adv(chunks: list, max_chars: int = None) -> str:
    """Format chunks with numbered headers for citation."""
    max_chars = max_chars or CONFIG["max_context_chars"]
    parts = []
    for i, c in enumerate(chunks, 1):
        m = c["metadata"]
        header = (
            f"[{i}] {m.get('ticker','?')} {m.get('form_type','?')} "
            f"{m.get('filing_year','?')} | {m.get('company','')}"
        )
        parts.append(f"{header}\n{c['text']}")
    context = "\n\n---\n\n".join(parts)
    return context[:max_chars]


def generate_with_citations(query: str, chunks: list) -> str:
    """Generate an answer with [n] citations referencing the retrieved chunks."""
    context  = format_context_adv(chunks)
    user_msg = f"Question:\n{query}\n\nRetrieved context (cite [n] for each claim):\n{context}"
    return call_llm(
        messages=[
            {"role": "system", "content": GENERATOR_SYSTEM_ADV},
            {"role": "user",   "content": user_msg},
        ],
        temperature=CONFIG["temperature_generator"],
        max_tokens=CONFIG["max_tokens_answer"],
    )

## 12. Full Advanced RAG Pipeline

In [53]:
def advanced_rag(query: str, verbose: bool = False) -> dict:
    """
    Advanced RAG pipeline:
    1. Metadata extraction  -> filter to relevant company/year
    2. Query rewriting      -> align with financial document vocabulary
    3. Multi-query          -> generate additional query variants
    4. Hybrid retrieval     -> BM25 + Dense for each query variant
    5. RRF merge            -> combine all ranked lists
    6. Adjacent expansion   -> add neighboring chunks to recover split tables
    7. CrossEncoder rerank  -> accurate relevance re-scoring
    8. Context compression  -> keep only most relevant sentences
    9. Generation           -> answer with [n] citations
    """
    # 1. Metadata extraction
    meta = extract_metadata(query)
    if verbose:
        print(f"  [Meta]     {meta}")

    # 2. Query rewriting
    rewritten = rewrite_query(query)
    if verbose:
        print(f"  [Rewrite]  {rewritten}")

    # 3. Multi-query generation
    variants    = generate_multi_queries(rewritten)
    all_queries = [rewritten] + variants
    if verbose:
        print(f"  [Queries]  {all_queries}")

    # 4. Hybrid retrieval (BM25 + Dense) for each query variant
    bm25_mask    = build_bm25_mask(**meta)
    ranked_lists = []
    for q in all_queries:
        ranked_lists.append(bm25_search(q, CONFIG["bm25_top_k"], bm25_mask))
        ranked_lists.append(dense_search(q, CONFIG["dense_top_k"], **meta))

    # 5. RRF merge
    candidates = rrf_merge(ranked_lists)
    if verbose:
        print(f"  [RRF]      {len(candidates)} unique candidates after merge")

    # 6. Adjacent expansion — add chunk_index ± 1 within the same filing
    candidates = expand_adjacent(candidates, expand_n=1)
    if verbose:
        print(f"  [Expand]   {len(candidates)} candidates after adjacent expansion")

    # 7. CrossEncoder reranking
    reranked = rerank(candidates, rewritten)
    if verbose:
        print(f"  [Rerank]   top {len(reranked)} chunks selected")

    # 8. Context compression
    compressed = compress_context(reranked, rewritten)

    # 9. Generation with citations
    context = format_context_adv(compressed)
    answer  = generate_with_citations(query, compressed)

    return {
        "query":      query,
        "rewritten":  rewritten,
        "variants":   variants,
        "metadata":   meta,
        "answer":     answer,
        "chunks":     reranked,
        "context":    context,
        "n_chunks":   len(reranked),
    }

### Quick Demo

In [54]:
demo_q = "What was Apple's total net revenue for fiscal year 2024?"
result = advanced_rag(demo_q, verbose=True)

print()
print("QUESTION:", result["query"])
print()
print("ANSWER:", result["answer"])
print()
print(f"Metadata extracted : {result['metadata']}")
print(f"Rewritten query    : {result['rewritten']}")
print(f"Query variants     : {result['variants']}")
print()
print(f"Retrieved {result['n_chunks']} chunks:")
for i, c in enumerate(result["chunks"], 1):
    m = c["metadata"]
    score = c.get("rerank_score", c.get("score", 0))
    compressed = "(compressed)" if c.get("compressed") else ""
    print(f"  [{i}] {m.get('ticker','?'):6s}  {m.get('form_type','?'):5s}  "
          f"year={m.get('filing_year','?')}  rerank={score:.3f}  {compressed}")

  [Meta]     {'ticker': 'AAPL', 'filing_year': 2024, 'form_type': None}
  [Rewrite]  Apple net revenue fiscal year 2024
  [Queries]  ['Apple net revenue fiscal year 2024', 'Apple total sales revenue 2024 fiscal year SEC filings', 'Apple income statement revenue FY24']
  [RRF]      42 unique candidates after merge
  [Expand]   73 candidates after adjacent expansion
  [Rerank]   top 7 chunks selected

QUESTION: What was Apple's total net revenue for fiscal year 2024?

ANSWER: Apple's total net revenue for fiscal year 2024 was $391,035 million [6].

Metadata extracted : {'ticker': 'AAPL', 'filing_year': 2024, 'form_type': None}
Rewritten query    : Apple net revenue fiscal year 2024
Query variants     : ['Apple total sales revenue 2024 fiscal year SEC filings', 'Apple income statement revenue FY24']

Retrieved 7 chunks:
  [1] AAPL    10-Q   year=2024  rerank=8.386  
  [2] AAPL    10-Q   year=2024  rerank=7.750  
  [3] AAPL    10-Q   year=2024  rerank=6.672  
  [4] AAPL    10-Q   year=2024

## 13. Evaluation (LLM-as-Judge)

Same judge prompts as Baseline RAG for fair comparison.
Results are saved to `./results/advanced_results.csv`.

In [55]:
CONFIG["eval_path"]

'../sec_rag_team_share/evaluation/GenAI Eval QA.csv'

In [56]:
eval_df = pd.read_csv(CONFIG["eval_path"])
eval_df = eval_df[eval_df["split"] == CONFIG["eval_split"]].reset_index(drop=True)

print(f"Evaluation split : '{CONFIG['eval_split']}'  ({len(eval_df)} questions)")
print(eval_df["question_type"].value_counts().to_string())

Evaluation split : 'dev'  (20 questions)
question_type
extractive     5
paraphrased    5
reasoning      5
adversarial    5


In [57]:
results = []

for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating advanced"):
    question          = str(row["question"])
    reference_answer  = str(row["expected_answer"]).strip()
    question_id       = row.get("question_id", idx)
    question_type     = row.get("question_type", "unknown")
    company           = row.get("company", "")
    ticker            = row.get("ticker", "")
    form_type         = row.get("form_type", "")
    filing_year       = row.get("filing_year", None)
    expected_decision = str(row.get("expected_decision", "answer")).lower().strip()

    print(f'\nQ{idx+1}/{len(eval_df)} [{question_type}]  {ticker}  {filing_year}')

    # ── Reset token accumulator ────────────────────────────────────────────────
    _reset_question_tokens()

    # ── Run pipeline ──────────────────────────────────────────────────────────
    t0 = time.time()
    rag = advanced_rag(question)
    latency_sec  = round(time.time() - t0, 2)
    final_answer = rag["answer"]
    context      = rag["context"]

    # ── Heuristic scores ───────────────────────────────────────────────────────
    t_f1    = token_f1_score(final_answer, reference_answer) if reference_answer else None
    num_acc = numerical_accuracy_score(final_answer, reference_answer) if reference_answer else None
    predicted_decision = compute_decision_from_text(final_answer)
    decision_accuracy  = 1.0 if predicted_decision == expected_decision else 0.0

    # ── LLM Judge ──────────────────────────────────────────────────────────────
    c_score, c_covered, c_reason = llm_judge_correctness(question, reference_answer, final_answer)
    f_score, _,         f_reason = llm_judge_faithfulness(question, context, final_answer)

    # ── Token / cost summary ───────────────────────────────────────────────────
    # 'generate' accumulates all call_llm calls (rewrite + multi-query + generation)
    gen_tok   = _question_tokens.get('generate',          {'input': 0, 'output': 0})
    corr_tok  = _question_tokens.get('judge_correctness', {'input': 0, 'output': 0})
    faith_tok = _question_tokens.get('judge_faithfulness',{'input': 0, 'output': 0})
    total_in, total_out = _get_question_token_totals()
    est_cost = _estimate_cost_usd(total_in, total_out)

    f1_str = f"{t_f1:.2f}" if t_f1 is not None else "N/A"
    print(f'  corr={c_score:.2f}  faith={f_score:.2f}  f1={f1_str}  '
          f'tokens={total_in}in/{total_out}out  est=${est_cost:.5f}')

    results.append({
        # Question metadata
        "question_id":              question_id,
        "question":                 question,
        "question_type":            question_type,
        "company":                  company,
        "ticker":                   ticker,
        "form_type":                form_type,
        "filing_year":              filing_year,
        "reference_answer":         reference_answer,
        "expected_decision":        expected_decision,
        # Answer
        "final_answer":             final_answer,
        "pipeline":                 "advanced_rag",
        # Pipeline metadata
        "n_chunks":                 rag["n_chunks"],
        "rewritten_query":          rag["rewritten"],
        "latency_sec":              latency_sec,
        # Heuristic scores
        "token_f1":                 t_f1,
        "numerical_accuracy":       num_acc,
        "decision_accuracy":        decision_accuracy,
        "predicted_decision":       predicted_decision,
        # LLM Judge
        "llm_correctness_score":    c_score,
        "llm_claims_covered":       c_covered,
        "llm_correctness_reason":   c_reason,
        "llm_faithfulness_score":   f_score,
        "llm_faithfulness_reason":  f_reason,
        # Model logging
        "model_generator":          LLM_MODEL,
        "model_judge_correctness":  CONFIG["judge_model"],
        "model_judge_faithfulness": CONFIG["judge_model"],
        # Token & cost (generate covers rewrite + multi-query + generation)
        "tokens_generate_input":    gen_tok["input"],
        "tokens_generate_output":   gen_tok["output"],
        "tokens_judge_corr_input":  corr_tok["input"],
        "tokens_judge_corr_output": corr_tok["output"],
        "tokens_judge_faith_input": faith_tok["input"],
        "tokens_judge_faith_output":faith_tok["output"],
        "tokens_total_input":       total_in,
        "tokens_total_output":      total_out,
        "est_cost_usd":             est_cost,
    })
    time.sleep(CONFIG["inter_question_sleep_sec"])

results_df = pd.DataFrame(results)
Path(CONFIG["results_dir"]).mkdir(parents=True, exist_ok=True)
out_path = Path(CONFIG["results_dir"]) / "advanced_results.csv"
results_df.to_csv(out_path, index=False)
print(f"\nSaved {len(results_df)} rows -> {out_path}")

Evaluating advanced:   0%|          | 0/20 [00:00<?, ?it/s]


Q1/20 [extractive]  AAPL  2023.0
  corr=1.00  faith=1.00  f1=0.78  tokens=7763in/219out  est=$0.00086


Evaluating advanced:   5%|▌         | 1/20 [00:53<16:59, 53.65s/it]


Q2/20 [extractive]  AAPL  2025.0
  corr=0.90  faith=1.00  f1=0.62  tokens=4022in/278out  est=$0.00051


Evaluating advanced:  10%|█         | 2/20 [01:44<15:32, 51.80s/it]

...


Q19/20 [adversarial]  MSFT  nan
  corr=0.00  faith=0.00  f1=0.00  tokens=9667in/208out  est=$0.00105


Evaluating advanced:  95%|█████████▌| 19/20 [16:47<00:55, 55.01s/it]


Q20/20 [adversarial]  NVDA  nan
  corr=0.00  faith=1.00  f1=0.00  tokens=5649in/201out  est=$0.00065


Evaluating advanced: 100%|██████████| 20/20 [17:39<00:00, 53.00s/it]


Saved 20 rows -> results\advanced_results.csv


In [58]:
results_df

,question_id,question,question_type,company,ticker,form_type,filing_year,reference_answer,expected_decision,final_answer,...,model_judge_faithfulness,tokens_generate_input,tokens_generate_output,tokens_judge_corr_input,tokens_judge_corr_output,tokens_judge_faith_input,tokens_judge_faith_output,tokens_total_input,tokens_total_output,est_cost_usd
0,1.0,What were Apple’s total net sales in fiscal ye...,extractive,Apple,AAPL,10-K,2023.0,Apple's total net sales in fiscal year 2023 we...,answer,Apple's total net sales were $383.3 billion in...,...,gemini-2.5-flash-lite,3879,77,173,72,3711,70,7763,219,0.000864
1,2.0,"According to Apple’s fiscal Q3 2025 filing, wh...",extractive,Apple,AAPL,10-Q,2025.0,"According to Apple's Q3 2025 filing, the U.S. ...",answer,"Beginning in the second quarter of 2025, new U...",...,gemini-2.5-flash-lite,1974,138,234,77,1814,63,4022,278,0.000513
2,3.0,What are the contingencies declared by Microso...,extractive,Microsoft,MSFT,10-Q,2023.0,Microsoft declared a contingency related to IR...,answer,"As of September 30, 2023, Microsoft had accrue...",...,gemini-2.5-flash-lite,4850,205,314,175,4791,63,9955,443,0.001173
3,4.0,Which Microsoft business segment generated the...,extractive,Microsoft,MSFT,10-K,2023.0,Intelligent Cloud generated the highest revenu...,answer,The Intelligent Cloud segment generated the hi...,...,gemini-2.5-flash-lite,5133,87,182,58,4969,62,10284,207,0.001111
4,5.0,What risk does Nvidia mention that could affec...,extractive,NVIDIA,NVDA,10-K,2025.0,Customers may delay adopting new architectures...,answer,Nvidia mentions that customers anticipating pr...,...,gemini-2.5-flash-lite,3192,122,199,120,3043,75,6434,317,0.000770
5,26.0,What amount of revenue did Apple generate duri...,paraphrased,Apple,AAPL,10-K,2023.0,Apple's total net sales in fiscal year 2023 we...,answer,"Apple generated $383,285 million in total net ...",...,gemini-2.5-flash-lite,3397,77,173,63,3230,53,6800,193,0.000757
6,27.0,Which regions were impacted by the newly annou...,paraphrased,Apple,AAPL,10-Q,2025.0,"According to Apple's Q3 2025 filing, the U.S. ...",answer,"The provided context states that ""several coun...",...,gemini-2.5-flash-lite,2712,140,227,77,2548,84,5487,301,0.000669
7,28.0,What potential financial obligations did Micro...,paraphrased,Microsoft,MSFT,10-Q,2023.0,Microsoft declared a contingency related to IR...,answer,"In the three-month period ending September 30,...",...,gemini-2.5-flash-lite,5761,153,255,123,5633,64,11649,340,0.001301
8,29.0,Which of Microsoft’s operating divisions contr...,paraphrased,Microsoft,MSFT,10-K,2023.0,Intelligent Cloud generated the highest revenu...,answer,"The ""Intelligent Cloud"" division contributed t...",...,gemini-2.5-flash-lite,4984,91,186,58,4817,57,9987,206,0.001081
9,30.0,What factor does Nvidia identify that could de...,paraphrased,NVIDIA,NVDA,10-K,2025.0,Customers may delay adopting new architectures...,answer,Nvidia identifies several factors that could d...,...,gemini-2.5-flash-lite,3118,223,302,114,3072,73,6492,410,0.000813


In [59]:
for idx, row in results_df.iterrows():
    print("Row:", idx)
    print("Question:", row['question'])
    print("Reference Answer:", row['reference_answer'])
    print("Final Answer:", row['final_answer'])
    # print("Generated Answer:", row['generated_answer'])
    print("Expected Decision:", row['expected_decision'])
    print("-" * 50)

Row: 0
Question: What were Apple’s total net sales in fiscal year 2023?
Reference Answer: Apple's total net sales in fiscal year 2023 were $383,285 million.
Final Answer: Apple's total net sales were $383.3 billion in fiscal year 2023 [1].
Expected Decision: answer
--------------------------------------------------
Row: 1
Question: According to Apple’s fiscal Q3 2025 filing, which countries were subject to new U.S. import tariffs?
Reference Answer: According to Apple's Q3 2025 filing, the U.S. imposed additional tariffs on imports from China, India, Japan, South Korea, Taiwan, Vietnam, and the European Union.
Final Answer: Beginning in the second quarter of 2025, new U.S. tariffs were announced on imports from China, India, Japan, South Korea, Taiwan, Vietnam, and the European Union (EU), among others [1, 2, 3, 5].
Expected Decision: answer
--------------------------------------------------
Row: 2
Question: What are the contingencies declared by Microsoft for the quarter ended in Septe

## 14. Results & Comparison with Baseline

In [60]:
results_df

,question_id,question,question_type,company,ticker,form_type,filing_year,reference_answer,expected_decision,final_answer,...,model_judge_faithfulness,tokens_generate_input,tokens_generate_output,tokens_judge_corr_input,tokens_judge_corr_output,tokens_judge_faith_input,tokens_judge_faith_output,tokens_total_input,tokens_total_output,est_cost_usd
0,1.0,What were Apple’s total net sales in fiscal ye...,extractive,Apple,AAPL,10-K,2023.0,Apple's total net sales in fiscal year 2023 we...,answer,Apple's total net sales were $383.3 billion in...,...,gemini-2.5-flash-lite,3879,77,173,72,3711,70,7763,219,0.000864
1,2.0,"According to Apple’s fiscal Q3 2025 filing, wh...",extractive,Apple,AAPL,10-Q,2025.0,"According to Apple's Q3 2025 filing, the U.S. ...",answer,"Beginning in the second quarter of 2025, new U...",...,gemini-2.5-flash-lite,1974,138,234,77,1814,63,4022,278,0.000513
2,3.0,What are the contingencies declared by Microso...,extractive,Microsoft,MSFT,10-Q,2023.0,Microsoft declared a contingency related to IR...,answer,"As of September 30, 2023, Microsoft had accrue...",...,gemini-2.5-flash-lite,4850,205,314,175,4791,63,9955,443,0.001173
3,4.0,Which Microsoft business segment generated the...,extractive,Microsoft,MSFT,10-K,2023.0,Intelligent Cloud generated the highest revenu...,answer,The Intelligent Cloud segment generated the hi...,...,gemini-2.5-flash-lite,5133,87,182,58,4969,62,10284,207,0.001111
4,5.0,What risk does Nvidia mention that could affec...,extractive,NVIDIA,NVDA,10-K,2025.0,Customers may delay adopting new architectures...,answer,Nvidia mentions that customers anticipating pr...,...,gemini-2.5-flash-lite,3192,122,199,120,3043,75,6434,317,0.000770
5,26.0,What amount of revenue did Apple generate duri...,paraphrased,Apple,AAPL,10-K,2023.0,Apple's total net sales in fiscal year 2023 we...,answer,"Apple generated $383,285 million in total net ...",...,gemini-2.5-flash-lite,3397,77,173,63,3230,53,6800,193,0.000757
6,27.0,Which regions were impacted by the newly annou...,paraphrased,Apple,AAPL,10-Q,2025.0,"According to Apple's Q3 2025 filing, the U.S. ...",answer,"The provided context states that ""several coun...",...,gemini-2.5-flash-lite,2712,140,227,77,2548,84,5487,301,0.000669
7,28.0,What potential financial obligations did Micro...,paraphrased,Microsoft,MSFT,10-Q,2023.0,Microsoft declared a contingency related to IR...,answer,"In the three-month period ending September 30,...",...,gemini-2.5-flash-lite,5761,153,255,123,5633,64,11649,340,0.001301
8,29.0,Which of Microsoft’s operating divisions contr...,paraphrased,Microsoft,MSFT,10-K,2023.0,Intelligent Cloud generated the highest revenu...,answer,"The ""Intelligent Cloud"" division contributed t...",...,gemini-2.5-flash-lite,4984,91,186,58,4817,57,9987,206,0.001081
9,30.0,What factor does Nvidia identify that could de...,paraphrased,NVIDIA,NVDA,10-K,2025.0,Customers may delay adopting new architectures...,answer,Nvidia identifies several factors that could d...,...,gemini-2.5-flash-lite,3118,223,302,114,3072,73,6492,410,0.000813


In [61]:
print("=" * 60)
print(" ADVANCED RAG — EVALUATION RESULTS")
print("=" * 60)

scored_c = results_df[results_df['llm_correctness_score'].notna()]
scored_f = results_df[results_df['llm_faithfulness_score'].notna()]
print(f'\nTotal questions    : {len(results_df)}')
print(f'Correctness judged : {len(scored_c)}')
print(f'Faithfulness judged: {len(scored_f)}')

print('\nOverall metrics:')
for col, label in [
    ('llm_correctness_score',  'LLM Correctness  '),
    ('llm_claims_covered',     'Claims Covered   '),
    ('llm_faithfulness_score', 'LLM Faithfulness '),
    ('token_f1',               'Token F1         '),
    ('numerical_accuracy',     'Numerical Accuracy'),
    ('decision_accuracy',      'Decision Accuracy'),
]:
    vals = results_df[col].dropna()
    if len(vals):
        print(f'  {label}: {vals.mean():.4f}  (n={len(vals)})')

print('\nBy question_type:')
type_cols = ['llm_correctness_score', 'llm_faithfulness_score', 'token_f1', 'decision_accuracy']
avail = [c for c in type_cols if c in results_df.columns]
if avail and 'question_type' in results_df.columns:
    print(results_df.groupby('question_type')[avail].mean().round(3).to_string())

if 'latency_sec' in results_df.columns:
    lat = results_df['latency_sec'].dropna()
    if len(lat):
        print(f'\nLatency: mean={lat.mean():.1f}s  median={lat.median():.1f}s  max={lat.max():.1f}s')

print('\nToken & cost summary:')
for col in ['tokens_generate_input', 'tokens_generate_output',
            'tokens_judge_corr_input', 'tokens_judge_corr_output',
            'tokens_judge_faith_input', 'tokens_judge_faith_output',
            'tokens_total_input', 'tokens_total_output']:
    if col in results_df.columns:
        print(f'  {col:35s}: {int(results_df[col].sum()):,}')
if 'est_cost_usd' in results_df.columns:
    print(f'  {"Total estimated cost (USD)":35s}: ${results_df["est_cost_usd"].sum():.4f}')

# Compare with baseline if available
baseline_path = Path(CONFIG["results_dir"]) / "baseline_results.csv"
if baseline_path.exists():
    baseline_df = pd.read_csv(baseline_path)
    print("\n" + "=" * 60)
    print(" COMPARISON: BASELINE vs ADVANCED RAG")
    print("=" * 60)
    cols = ['llm_correctness_score', 'llm_faithfulness_score', 'token_f1', 'decision_accuracy']
    avail = [c for c in cols if c in results_df.columns and c in baseline_df.columns]
    comparison = pd.DataFrame({
        "Baseline": baseline_df[avail].mean(),
        "Advanced": results_df[avail].mean(),
    }).round(3)
    comparison["Delta"] = (comparison["Advanced"] - comparison["Baseline"]).round(3)
    print(comparison.to_string())
else:
    print("\n(Run baseline_rag.ipynb first to see comparison)")

 ADVANCED RAG — EVALUATION RESULTS

Total questions    : 20
Correctness judged : 20
Faithfulness judged: 20

Overall metrics:
  LLM Correctness  : 0.4950  (n=20)
  Claims Covered   : 0.5015  (n=20)
  LLM Faithfulness : 0.8650  (n=20)
  Token F1         : 0.3058  (n=20)
  Numerical Accuracy: 0.6897  (n=13)
  Decision Accuracy: 0.9000  (n=20)

By question_type:
               llm_correctness_score  llm_faithfulness_score  token_f1  decision_accuracy
question_type                                                                            
adversarial                     0.00                    0.60     0.000                0.8
extractive                      0.82                    1.00     0.536                1.0
paraphrased                     0.46                    1.00     0.387                1.0
reasoning                       0.70                    0.86     0.300                0.8

Latency: mean=11.1s  median=11.0s  max=16.9s

Token & cost summary:
  tokens_generate_input      